# World Cup 2026 Analytics

This notebook is the final analytical layer of the project. It focuses on understanding historical international football results and preparing evidence-based insights that can later support World Cup 2026 discussions.

The primary data source is the PostgreSQL table `matches`. If the database connection is unavailable, the notebook automatically falls back to `data/interim/matches_standardized.csv`, so the analysis remains reproducible.


## 1. Introduction

**Project goal.** Explore historical national-team matches to understand long-run performance, home advantage, scoring patterns, and tournament structure.

**Data source.** Standardized international football results from `results.csv` of the `martj42/international_results` repository, already loaded into the local project database and preserved in the interim CSV.

**Dataset volume.** The current analytical dataset contains **49,520 matches** and spans **1872-11-30 to 2026-07-19**.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'interim' / 'matches_standardized.csv'
ENV_PATH = PROJECT_ROOT / '.env'

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['figure.dpi'] = 120

PRIMARY_COLOR = '#0B6E4F'
SECONDARY_COLOR = '#F4A259'
ACCENT_COLOR = '#5B8E7D'
NEUTRAL_COLOR = '#5C677D'


## 2. Load Data

In [ ]:
def load_env_file(env_path: Path) -> dict[str, str]:
    """Read key-value pairs from a simple .env file."""
    values: dict[str, str] = {}
    if not env_path.exists():
        return values

    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        values[key.strip()] = value.strip()
    return values


def build_database_url(env: dict[str, str]) -> str | None:
    """Build a PostgreSQL URL from environment values when possible."""
    required_keys = ['DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']
    if not all(env.get(key) for key in required_keys):
        return None

    return (
        f"postgresql+psycopg://{env['DB_USER']}:{env['DB_PASSWORD']}"
        f"@{env['DB_HOST']}:{env['DB_PORT']}/{env['DB_NAME']}"
    )


def load_matches_dataframe() -> tuple[pd.DataFrame, str]:
    """Load match data from PostgreSQL and fall back to the interim CSV if needed."""
    env = dict(os.environ)
    env.update(load_env_file(ENV_PATH))
    database_url = build_database_url(env)

    sql_query = text(
        """
        SELECT
            match_date AS date,
            home_team,
            away_team,
            home_score,
            away_score,
            tournament,
            city,
            country,
            neutral
        FROM matches
        ORDER BY match_date
        """
    )

    if database_url:
        try:
            engine = create_engine(database_url)
            with engine.connect() as connection:
                df = pd.read_sql(sql_query, connection)
            return df, 'PostgreSQL table: matches'
        except Exception as error:
            print(f'PostgreSQL connection failed, using CSV fallback. Reason: {error}')

    df = pd.read_csv(DATA_PATH)
    return df, f'CSV fallback: {DATA_PATH.relative_to(PROJECT_ROOT)}'


matches, data_source = load_matches_dataframe()
matches['date'] = pd.to_datetime(matches['date'])
matches['home_score'] = pd.to_numeric(matches['home_score'])
matches['away_score'] = pd.to_numeric(matches['away_score'])
matches['neutral'] = matches['neutral'].astype('boolean')

matches['year'] = matches['date'].dt.year
matches['total_goals'] = matches['home_score'] + matches['away_score']
matches['goal_diff'] = (matches['home_score'] - matches['away_score']).abs()
matches['result'] = matches.apply(
    lambda row: 'Home win'
    if row['home_score'] > row['away_score']
    else 'Away win'
    if row['home_score'] < row['away_score']
    else 'Draw',
    axis=1,
)

print(f'Data source: {data_source}')
display(matches.head())


In [ ]:
print('Shape:', matches.shape)
print()
print('Info:')
matches.info()
print()
print('Missing values:')
display(matches.isna().sum().to_frame('missing_values'))
print()
print('First rows:')
display(matches.head())


## 3. Dataset Overview

In [ ]:
overview = pd.DataFrame(
    {
        'Metric': ['Matches', 'Teams', 'Tournaments', 'Date range'],
        'Value': [
            f"{len(matches):,}",
            f"{pd.unique(pd.concat([matches['home_team'], matches['away_team']])).size:,}",
            f"{matches['tournament'].nunique():,}",
            f"{matches['date'].min().date()} to {matches['date'].max().date()}",
        ],
    }
)

display(overview)


The dataset is historically broad rather than tournament-specific. It combines friendlies, qualifiers, and final tournaments across more than 150 years of international football. That scale makes it suitable for both descriptive analysis and later feature engineering.


## 4. Visualizations

In [ ]:
home_view = matches[['home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'goals_for', 'away_score': 'goals_against'}
)
away_view = matches[['away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'goals_for', 'home_score': 'goals_against'}
)
team_long = pd.concat([home_view, away_view], ignore_index=True)
team_long['win'] = (team_long['goals_for'] > team_long['goals_against']).astype(int)
team_long['draw'] = (team_long['goals_for'] == team_long['goals_against']).astype(int)
team_long['loss'] = (team_long['goals_for'] < team_long['goals_against']).astype(int)

team_summary = (
    team_long.groupby('team', as_index=False)
    .agg(
        matches=('team', 'size'),
        wins=('win', 'sum'),
        draws=('draw', 'sum'),
        losses=('loss', 'sum'),
        goals_for=('goals_for', 'sum'),
        goals_against=('goals_against', 'sum'),
    )
)
team_summary['goal_diff'] = team_summary['goals_for'] - team_summary['goals_against']
team_summary['win_rate'] = team_summary['wins'] / team_summary['matches'] * 100
team_summary['avg_goals_for'] = team_summary['goals_for'] / team_summary['matches']

matches_by_year = matches.groupby('year', as_index=False).agg(matches=('date', 'size'))
avg_goals_by_year = matches.groupby('year', as_index=False).agg(avg_total_goals=('total_goals', 'mean'))


### 4.1 Matches by Year

In [ ]:
fig, ax = plt.subplots()
ax.plot(matches_by_year['year'], matches_by_year['matches'], color=PRIMARY_COLOR, linewidth=2.2)
ax.set_title('International Matches by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Number of matches')
ax.grid(alpha=0.25)
plt.show()


International football grew from an occasional event into a dense global calendar. The sharp long-run rise is one of the clearest patterns in the whole project, and the peak in this dataset comes in 2024. Short-term dips exist, but they do not change the broader expansion trend.


### 4.2 Average Total Goals by Year

In [ ]:
fig, ax = plt.subplots()
ax.plot(avg_goals_by_year['year'], avg_goals_by_year['avg_total_goals'], color=SECONDARY_COLOR, linewidth=2)
ax.set_title('Average Total Goals per Match by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Average goals')
ax.grid(alpha=0.25)
plt.show()


Scoring was much more volatile in the early decades, when yearly samples were small and some seasons were extremely high-scoring. In the modern period, the average usually stays in a tighter band around roughly 2.5 to 3.0 goals per match. That stability makes later-era comparisons more reliable than very early historical comparisons.


### 4.3 Top-20 Teams by Total Wins

In [ ]:
top_wins = team_summary.sort_values(['wins', 'win_rate'], ascending=[False, False]).head(20)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_wins['team'], top_wins['wins'], color=PRIMARY_COLOR)
ax.set_title('Top-20 National Teams by Total Wins')
ax.set_xlabel('Wins')
ax.set_ylabel('Team')
ax.invert_yaxis()
plt.show()


Brazil leads the dataset by total wins, followed by England, Germany, and Argentina. The top of the ranking mixes elite performance with long historical participation, so high rank usually reflects both quality and very large match volume. This is why historical mainstays such as Sweden and South Korea also appear near the top.


### 4.4 Top-20 Teams by Win Rate (Minimum 100 Matches)

In [ ]:
top_win_rate = (
    team_summary.loc[team_summary['matches'] >= 100]
    .sort_values(['win_rate', 'wins'], ascending=[False, False])
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_win_rate['team'], top_win_rate['win_rate'], color=ACCENT_COLOR)
ax.set_title('Top-20 Teams by Win Rate (Minimum 100 Matches)')
ax.set_xlabel('Win rate, %')
ax.set_ylabel('Team')
ax.invert_yaxis()
plt.show()


The 100-match filter removes very small samples and makes the ranking more defensible. Brazil remains near the very top even under that stricter rule, which strengthens its case as the most consistently successful team in the dataset. Smaller but still substantial football systems such as Jersey and Guernsey also stand out on raw win rate.


### 4.5 Highest-Scoring Teams

In [ ]:
top_scoring = (
    team_summary.loc[team_summary['matches'] >= 100]
    .sort_values(['avg_goals_for', 'goals_for'], ascending=[False, False])
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_scoring['team'], top_scoring['avg_goals_for'], color=SECONDARY_COLOR)
ax.set_title('Top-20 Teams by Average Goals Scored (Minimum 100 Matches)')
ax.set_xlabel('Average goals scored per match')
ax.set_ylabel('Team')
ax.invert_yaxis()
plt.show()


This ranking looks different from the pure wins table because it rewards offensive output rather than match outcomes alone. Several regional teams with open, high-scoring match histories rank very highly, while Germany, England, and Brazil show that elite teams can combine scale with strong attacking production. It is a useful reminder that win rate and scoring rate capture different aspects of performance.


### 4.6 Distribution of Total Goals

In [ ]:
fig, ax = plt.subplots()
ax.hist(matches['total_goals'], bins=range(0, int(matches['total_goals'].max()) + 2), color=NEUTRAL_COLOR, edgecolor='white')
ax.set_title('Distribution of Total Goals per Match')
ax.set_xlabel('Total goals in a match')
ax.set_ylabel('Number of matches')
ax.set_xlim(0, 12)
plt.show()


Most international matches cluster in a low-to-mid scoring range. The center of the distribution sits around 2 to 3 total goals, while very high-scoring games are clearly rare tail events. That shape supports the earlier SQL finding that extreme scores exist, but they represent outliers rather than the norm.


### 4.7 Home Wins, Draws, and Away Wins

In [ ]:
result_distribution = (
    matches['result']
    .value_counts(normalize=True)
    .reindex(['Home win', 'Draw', 'Away win'])
    .mul(100)
)

fig, ax = plt.subplots()
ax.bar(result_distribution.index, result_distribution.values, color=[PRIMARY_COLOR, NEUTRAL_COLOR, SECONDARY_COLOR])
ax.set_title('Match Outcome Distribution')
ax.set_xlabel('Result')
ax.set_ylabel('Share of matches, %')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
plt.show()


Home wins are the single most common result in the dataset, well ahead of away wins. Draws form a substantial but smaller share, while away victories are the least frequent of the three categories. Even before separating neutral venues, the aggregate pattern already suggests a meaningful home advantage.


### 4.8 Neutral Venues vs Non-Neutral Venues

In [ ]:
venue_comparison = (
    matches.assign(
        venue_type=matches['neutral'].map({True: 'Neutral', False: 'Non-neutral'}).fillna('Unknown')
    )
    .groupby(['venue_type', 'result'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['Home win', 'Draw', 'Away win'])
)
venue_share = venue_comparison.div(venue_comparison.sum(axis=1), axis=0).mul(100)

fig, ax = plt.subplots()
venue_share.plot(kind='bar', stacked=True, ax=ax, color=[PRIMARY_COLOR, NEUTRAL_COLOR, SECONDARY_COLOR])
ax.set_title('Outcome Shares: Neutral vs Non-Neutral Matches')
ax.set_xlabel('Venue type')
ax.set_ylabel('Share of matches, %')
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.legend(title='Result', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()


The non-neutral group shows the clearest home advantage: home teams win just over half of these matches. On neutral fields, the home side still wins more often than it loses, but the gap narrows and away wins become more common. This is one of the strongest and most practically useful findings in the project.


### 4.9 Top Tournaments by Match Count

In [ ]:
top_tournaments = matches['tournament'].value_counts().head(15).sort_values()

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_tournaments.index, top_tournaments.values, color=ACCENT_COLOR)
ax.set_title('Top Tournaments by Number of Matches')
ax.set_xlabel('Matches')
ax.set_ylabel('Tournament')
plt.show()


Friendlies dominate the dataset by a very wide margin, which is important when interpreting any global average. Qualification tournaments form the next major block, while final tournaments are much smaller by volume. In other words, the historical database is driven more by regular international scheduling than by flagship finals alone.


### 4.10 Top Host Countries

In [ ]:
top_countries = matches['country'].value_counts().head(15).sort_values()

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_countries.index, top_countries.values, color=NEUTRAL_COLOR)
ax.set_title('Top Host Countries by Number of Matches')
ax.set_xlabel('Matches hosted')
ax.set_ylabel('Country')
plt.show()


The United States is the most frequent host country in this dataset, followed by countries such as France and Malaysia. This ranking reflects a mix of football tradition, tournament hosting, and the concentration of regional competitions. It also reminds us that the location dimension in the data is rich enough for event-hosting analysis, not only team-performance analysis.


## 5. Interesting Insights

The SQL analysis stage produced several compact findings that are worth surfacing again in notebook form:

- The dataset spans **49,520 matches** across **337 teams**, **201 tournaments**, and **269 host countries**.
- The earliest recorded match is **Scotland 0:0 England on 1872-11-30**.
- The most active year in the dataset is **2024**, with **1,231 matches**.
- Among years with at least 50 matches, **1912** has the highest average scoring rate at **5.06 goals per match**.
- **Brazil** leads the historical ranking by total wins and also remains elite by win rate among teams with large sample sizes.
- **Friendly** matches dominate the database by volume, with **18,387** matches.
- The **United States** is the most frequent host country, with **1,585** matches.
- The most frequent host city is **Kuala Lumpur**, with **745** matches.
- The biggest home win in the data is **Australia 31:0 American Samoa** on **2001-04-11**.
- The biggest away win is **Guam 0:21 North Korea** on **2005-03-11**.
- The most common scoreline is **1:0**, observed **5,106** times.
- There are **9 matches with scores above 20 goals**, and they should be treated as historical outliers that require verification rather than automatic deletion.


## 6. Conclusion

This notebook confirms that the project has already reached a meaningful analytical stage. Using only historical match records, we can clearly see the long-run growth of international football, the persistence of home advantage, the concentration of success among a relatively small group of teams, and the importance of friendlies and qualification matches in shaping the full dataset.

The data is strong for descriptive and comparative analysis, but it also has limitations. It is not a player-level dataset, it does not explain tactical context, and some categorical fields contain visible text-encoding issues. In addition, unusual scorelines are historically important but should be reviewed carefully before being used in public-facing outputs.

The most natural next analytical directions are feature engineering, form-based performance windows, tournament-specific comparisons, and simple predictive modeling of match outcomes. Those steps should build on the same reproducible foundation without altering the raw historical record.
